# Data Preprocess

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>step</th>
      <th>feature</th>
      <th>action</th>
      <th>reason</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>1</td>
      <td>DAYS_EMPLOYED</td>
      <td>365243 -&gt; NaN</td>
      <td>365243 is an abnormal sentinel value</td>
    </tr>
    <tr>
      <th>1</th>
      <td>2</td>
      <td>DAYS_EMPLOYED_ANOM</td>
      <td>create 0/1 flag</td>
      <td>keep information about the original sentinel</td>
    </tr>
    <tr>
      <th>2</th>
      <td>3</td>
      <td>numeric columns</td>
      <td>inf -&gt; NaN</td>
      <td>avoid infinite values in later modeling</td>
    </tr>
    <tr>
      <th>3</th>
      <td>4</td>
      <td>missing values</td>
      <td>keep as missing</td>
      <td>imputation should depend on the model</td>
    </tr>
    <tr>
      <th>4</th>
      <td>5</td>
      <td>categorical columns</td>
      <td>keep original categories</td>
      <td>encoding should depend on the model</td>
    </tr>
    <tr>
      <th>5</th>
      <td>6</td>
      <td>extreme values / rare categories</td>
      <td>audit only</td>
      <td>do not remove values without clear evidence</td>
    </tr>
  </tbody>
</table>
</div>

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")

TRAIN_FILE = DATA_DIR / "application_train.csv"
TEST_FILE = DATA_DIR / "application_test.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR   :", DATA_DIR.resolve())
print("OUTPUT_DIR :", OUTPUT_DIR.resolve())

DATA_DIR   : /Users/ethan/Code Repository/MSDM 5054/Project/Project 1/data/raw
OUTPUT_DIR : /Users/ethan/Code Repository/MSDM 5054/Project/Project 1/data/processed


## 1. Load Data

In [2]:
raw_train = pd.read_csv(TRAIN_FILE)
raw_test = pd.read_csv(TEST_FILE)

print("raw_train:", raw_train.shape)
print("raw_test :", raw_test.shape)

raw_train.head()

raw_train: (307511, 122)
raw_test : (48744, 121)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,"202,500.0000","406,597.5000","24,700.5000","351,000.0000",Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.0188,-9461,-637,"-3,648.0000",-2120,NaN,1,1,0,1,1,0,Laborers,1.0000,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.0830,0.2629,0.1394,0.0247,0.0369,0.9722,0.6192,0.0143,0.0000,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.0220,0.0198,0.0000,0.0000,0.0250,0.0369,0.9722,0.6243,0.0144,0.0000,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.0000,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0000,2.0000,2.0000,2.0000,"-1,134.0000",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,100003,0,Cash loans,F,N,N,0,"270,000.0000","1,293,502.5000","35,698.5000","1,129,500.0000",Family,State servant,Higher education,Married,House / apartment,0.0035,-16765,-1188,"-1,186.0000",-291,NaN,1,1,0,1,1,0,Core staff,2.0000,1,1,MONDAY,11,0,0,0,0,0,0,School,0.3113,0.6222,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.0800,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.0790,0.0554,0.0000,0.0000,0.0968,0.0529,0.9851,0.7987,0.0608,0.0800,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.0100,reg oper account,block of flats,0.0714,Block,No,1.0000,0.0000,1.0000,0.0000,-828.0000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,100004,0,Revolving loans,M,Y,Y,0,"67,500.0000","135,000.0000","6,750.0000","135,000.0000",Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.0100,-19046,-225,"-4,260.0000",-2531,26.0000,1,1,1,1,1,0,Laborers,1.0000

- 指定 ID 列 SK_ID_CURR 和目标变量 TARGET。
- 检查训练集是否包含 TARGET，测试集是否不包含 TARGET。
- 检查训练集和测试集是否存在完全重复的行。
- 检查 SK_ID_CURR 是否重复，确认 application 主表是否保持“一位客户一行”。
- 检查 train 和 test 的客户 ID 是否有交集，避免同一客户同时出现在两个数据集中。
- 检查除 TARGET 外，train 和 test 的原始特征列是否一致。
- 检查 TARGET 实际包含哪些取值。

In [3]:
ID_COL = "SK_ID_CURR"
TARGET_COL = "TARGET"

print("Train has TARGET:", TARGET_COL in raw_train.columns)
print("Test has TARGET :", TARGET_COL in raw_test.columns)

print("Duplicate train rows:", raw_train.duplicated().sum())
print("Duplicate test rows:", raw_test.duplicated().sum())
print("Duplicate train IDs:", raw_train[ID_COL].duplicated().sum())
print("Duplicate test IDs:", raw_test[ID_COL].duplicated().sum())

id_overlap = len(set(raw_train[ID_COL]) & set(raw_test[ID_COL]))
print("Train/test ID overlap:", id_overlap)

train_feature_cols_raw = [
    col for col in raw_train.columns
    if col != TARGET_COL
]
print("Train/test columns match:", train_feature_cols_raw == list(raw_test.columns))
print("TARGET values:", sorted(raw_train[TARGET_COL].dropna().unique()))


Train has TARGET: True
Test has TARGET : False
Duplicate train rows: 0
Duplicate test rows: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train/test ID overlap: 0
Train/test columns match: True
TARGET values: [np.int64(0), np.int64(1)]


In [4]:
data_overview = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(raw_train), len(raw_test)],
    "columns": [raw_train.shape[1], raw_test.shape[1]],
    "unique_SK_ID_CURR": [
        raw_train["SK_ID_CURR"].nunique(),
        raw_test["SK_ID_CURR"].nunique()
    ],
    "missing_cells": [
        int(raw_train.isna().sum().sum()),
        int(raw_test.isna().sum().sum())
    ],
    "target_positive_rate": [raw_train[TARGET_COL].mean(), np.nan],
})

display(data_overview)

,dataset,rows,columns,unique_SK_ID_CURR,missing_cells,target_positive_rate
0,train,307511,122,307511,9152465,0.0807
1,test,48744,121,48744,1404419,NaN


## 2. Quatify the `DAYS_EMPLOYED` sentinel

检查 DAYS_EMPLOYED 中的异常占位值
- 将 365243 定义为 DAYS_EMPLOYED 的异常占位值（sentinel value）。
- 分别统计 train 和 test 中这个值出现的次数和比例。
- 同时统计清洗前 DAYS_EMPLOYED 已经存在的缺失值数量。
- 将结果整理到 sentinel_before 表中，作为正式替换前的记录。

In [5]:
EMPLOYED_SENTINEL = 365243

sentinel_before = pd.DataFrame({
    "dataset": ["train", "test"],
    "sentinel_count": [
        (raw_train["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum(),
        (raw_test["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum()
    ],
    "sentinel_rate": [
        (raw_train["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).mean(),
        (raw_test["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).mean()
    ],
    "missing_before": [
        raw_train["DAYS_EMPLOYED"].isna().sum(),
        raw_test["DAYS_EMPLOYED"].isna().sum()
    ]
})

sentinel_before

,dataset,sentinel_count,sentinel_rate,missing_before
0,train,55374,0.1801,0
1,test,9274,0.1903,0


## 3. Clean the application data

In [6]:
def clean_application(df):
    df = df.copy()

    df["DAYS_EMPLOYED_ANOM"] = (
        df["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL
    ).astype(int)

    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(
        EMPLOYED_SENTINEL,
        np.nan
    )

    df = df.replace([np.inf, -np.inf], np.nan)

    return df

In [7]:
train = clean_application(raw_train)
test = clean_application(raw_test)

print("clean train:", train.shape)
print("clean test :", test.shape)

clean train: (307511, 123)
clean test : (48744, 122)


In [8]:
sentinel_check = pd.DataFrame({
    "dataset": ["train", "test"],
    "raw_sentinel_count": [
        (raw_train["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum(),
        (raw_test["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum()
    ],
    "clean_sentinel_count": [
        (train["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum(),
        (test["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum()
    ],
    # anomaly_flag_count should equal to raw_sentinel_count
    "anomaly_flag_count": [
        train["DAYS_EMPLOYED_ANOM"].sum(),
        test["DAYS_EMPLOYED_ANOM"].sum()
    ]
})

sentinel_check

,dataset,raw_sentinel_count,clean_sentinel_count,anomaly_flag_count
0,train,55374,0,55374
1,test,9274,0,9274


## 4. Missing value analysis

In [9]:
raw_missing = raw_train.isna().sum()
clean_missing = train.reindex(columns=raw_train.columns).isna().sum()

missing_change_train = pd.DataFrame({
    "raw_missing": raw_missing,
    "clean_missing": clean_missing
})

missing_change_train["change"] = (
    missing_change_train["clean_missing"]
    - missing_change_train["raw_missing"]
)

print("Columns whose missing counts changed after cleaning:")
display(
    missing_change_train[
        missing_change_train["change"] != 0
    ]
)

missing_summary = pd.DataFrame({
    "train_missing_rate": train.isna().mean(),
    "test_missing_rate": test.reindex(columns=[
        col for col in train.columns
        if col != TARGET_COL
    ]).isna().mean()
})

display(missing_summary.sort_values("train_missing_rate", ascending=False).head(20))

Columns whose missing counts changed after cleaning:


,raw_missing,clean_missing,change
DAYS_EMPLOYED,0,55374,55374


,train_missing_rate,test_missing_rate
COMMONAREA_AVG,0.6987,0.6872
COMMONAREA_MEDI,0.6987,0.6872
COMMONAREA_MODE,0.6987,0.6872
NONLIVINGAPARTMENTS_MEDI,0.6943,0.6841
NONLIVINGAPARTMENTS_AVG,0.6943,0.6841
NONLIVINGAPARTMENTS_MODE,0.6943,0.6841
FONDKAPREMONT_MODE,0.6839,0.6728
LIVINGAPARTMENTS_MODE,0.6835,0.6725
LIVINGAPARTMENTS_MEDI,0.6835,0.6725
LIVINGAPARTMENTS_AVG,0.6835,0.6725


1. 只有 `DAYS_EMPLOYED` 的缺失数量发生了变化，这个增加量与训练集中原来的 365243 数量完全一致，因此这不是新的数据丢失，而是之前将异常占位值转换成 `NaN` 所导致的。
2. train/test 的缺失模式中，大多数高缺失特征的差异在1%～2%，但 `EXT_SOURCE_1` 比较突出，这说明 `EXT_SOURCE_1` 在 train 和 test 中的可用程度存在明显差异。

## 5. Categorical vaiable inspection

In [10]:
categorical_cols = (train.select_dtypes(include=["object", "string", "category"]).columns.tolist())

categorical_summary = pd.DataFrame({
    "feature": categorical_cols,
    "train_n_unique": [
        train[col].nunique()
        for col in categorical_cols
    ],
    "test_n_unique": [
        test[col].nunique()
        for col in categorical_cols
    ],
    "train_missing_rate": [
        train[col].isna().mean()
        for col in categorical_cols
    ],
    "test_missing_rate": [
        test[col].isna().mean()
        for col in categorical_cols
    ]
})

display(categorical_summary)

,feature,train_n_unique,test_n_unique,train_missing_rate,test_missing_rate
0,NAME_CONTRACT_TYPE,2,2,0.0000,0.0000
1,CODE_GENDER,3,2,0.0000,0.0000
2,FLAG_OWN_CAR,2,2,0.0000,0.0000
3,FLAG_OWN_REALTY,2,2,0.0000,0.0000
4,NAME_TYPE_SUITE,7,7,0.0042,0.0187
5,NAME_INCOME_TYPE,8,7,0.0000,0.0000
6,NAME_EDUCATION_TYPE,5,5,0.0000,0.0000
7,NAME_FAMILY_STATUS,6,5,0.0000,0.0000
8,NAME_HOUSING_TYPE,6,6,0.0000,0.0000
9,OCCUPATION_TYPE,18,18,0.3135,0.3201


In [11]:
special_tokens = ["XNA", "Unknown"]
special_rows = []

for col in categorical_cols:
    for token in special_tokens:
        train_count = (train[col] == token).sum()
        test_count = (test[col] == token).sum()

        if train_count > 0 or test_count > 0:
            special_rows.append({
                "feature": col,
                "token": token,
                "train_count": train_count,
                "test_count": test_count
            })

special_category_summary = pd.DataFrame(
    special_rows
)

display(special_category_summary)

,feature,token,train_count,test_count
0,CODE_GENDER,XNA,4,0
1,NAME_FAMILY_STATUS,Unknown,2,0
2,ORGANIZATION_TYPE,XNA,55374,9274


### Structural missingness

In [12]:
structural_missingness = pd.DataFrame({
    "pattern": [
        "OWN_CAR_AGE missing when FLAG_OWN_CAR = N",
        "ORGANIZATION_TYPE = XNA when DAYS_EMPLOYED_ANOM = 1"
    ],
    "train_rate": [
        train.loc[
            train["FLAG_OWN_CAR"] == "N",
            "OWN_CAR_AGE"
        ].isna().mean(),
        (
            train.loc[
                train["DAYS_EMPLOYED_ANOM"] == 1,
                "ORGANIZATION_TYPE"
            ] == "XNA"
        ).mean()
    ],
    "test_rate": [
        test.loc[
            test["FLAG_OWN_CAR"] == "N",
            "OWN_CAR_AGE"
        ].isna().mean(),
        (
            test.loc[
                test["DAYS_EMPLOYED_ANOM"] == 1,
                "ORGANIZATION_TYPE"
            ] == "XNA"
        ).mean()
    ]
})

display(structural_missingness)

,pattern,train_rate,test_rate
0,OWN_CAR_AGE missing when FLAG_OWN_CAR = N,1.0000,1.0000
1,ORGANIZATION_TYPE = XNA when DAYS_EMPLOYED_ANO...,1.0000,1.0000


## 6. Numerical extreme values inspection

In [13]:
feature_cols = [
    col for col in train.columns
    if col != TARGET_COL
]

numeric_feature_cols = [
    col for col in feature_cols
    if col != ID_COL
    and pd.api.types.is_numeric_dtype(train[col])
]

extreme_rows = []

for col in numeric_feature_cols:
    values = train[col].dropna()

    if len(values) == 0:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 3 * iqr
    upper = q3 + 3 * iqr

    outside = (
        (values < lower)
        | (values > upper)
    )

    extreme_rows.append({
        "feature": col,
        "min": values.min(),
        "median": values.median(),
        "max": values.max(),
        "outside_3iqr_count": outside.sum(),
        "outside_3iqr_rate": outside.mean()
    })

extreme_value_summary = pd.DataFrame(extreme_rows)

display(
    extreme_value_summary
    .sort_values("outside_3iqr_rate", ascending=False)
    .head(20)
)

,feature,min,median,max,outside_3iqr_count,outside_3iqr_rate
18,REGION_RATING_CLIENT,1.0000,2.0000,3.0000,80527,0.2619
19,REGION_RATING_CLIENT_W_CITY,1.0000,2.0000,3.0000,78027,0.2537
25,REG_CITY_NOT_WORK_CITY,0.0000,0.0000,1.0000,70867,0.2305
13,FLAG_WORK_PHONE,0.0000,0.0000,1.0000,61308,0.1994
102,AMT_REQ_CREDIT_BUREAU_QRT,0.0000,0.0000,261.0000,50575,0.1901
12,FLAG_EMP_PHONE,0.0000,1.0000,1.0000,55386,0.1801
104,DAYS_EMPLOYED_ANOM,0.0000,0.0000,1.0000,55374,0.1801
26,LIVE_CITY_NOT_WORK_CITY,0.0000,0.0000,1.0000,55215,0.1796
101,AMT_REQ_CREDIT_BUREAU_MON,0.0000,0.0000,27.0000,43759,0.1645
74,DEF_30_CNT_SOCIAL_CIRCLE,0.0000,0.0000,34.0000,35166,0.1147


In [14]:
train_feature_cols = [
    col for col in train.columns
    if col != TARGET_COL
]

test = test[train_feature_cols].copy()

FEATURE_COLS = [
    col for col in train.columns
    if col not in {ID_COL, TARGET_COL}
]

CATEGORICAL_FEATURES = [
    col for col in FEATURE_COLS
    if col in categorical_cols
]

NUMERIC_FEATURES = [
    col for col in FEATURE_COLS
    if col not in CATEGORICAL_FEATURES
]

print("Number of model features:", len(FEATURE_COLS))
print("Numeric features        :", len(NUMERIC_FEATURES))
print("Categorical features    :", len(CATEGORICAL_FEATURES))


Number of model features: 121
Numeric features        : 105
Categorical features    : 16


## 7. Export

In [15]:
preprocessing_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "unique_SK_ID_CURR",
        "missing_cells",
        "DAYS_EMPLOYED_sentinel",
        "DAYS_EMPLOYED_ANOM_sum"
    ],
    "raw_train": [
        len(raw_train),
        raw_train.shape[1],
        raw_train[ID_COL].nunique(),
        raw_train.isna().sum().sum(),
        (raw_train["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum(),
        np.nan
    ],
    "clean_train": [
        len(train),
        train.shape[1],
        train[ID_COL].nunique(),
        train.isna().sum().sum(),
        (train["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum(),
        train["DAYS_EMPLOYED_ANOM"].sum()
    ],
    "raw_test": [
        len(raw_test),
        raw_test.shape[1],
        raw_test[ID_COL].nunique(),
        raw_test.isna().sum().sum(),
        (raw_test["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum(),
        np.nan
    ],
    "clean_test": [
        len(test),
        test.shape[1],
        test[ID_COL].nunique(),
        test.isna().sum().sum(),
        (test["DAYS_EMPLOYED"] == EMPLOYED_SENTINEL).sum(),
        test["DAYS_EMPLOYED_ANOM"].sum()
    ]
})

preprocessing_summary


,metric,raw_train,clean_train,raw_test,clean_test
0,rows,"307,511.0000",307511,"48,744.0000",48744
1,columns,122.0000,123,121.0000,122
2,unique_SK_ID_CURR,"307,511.0000",307511,"48,744.0000",48744
3,missing_cells,"9,152,465.0000",9207839,"1,404,419.0000",1413693
4,DAYS_EMPLOYED_sentinel,"55,374.0000",0,"9,274.0000",0
5,DAYS_EMPLOYED_ANOM_sum,NaN,55374,NaN,9274


In [16]:
TRAIN_PARQUET = (
    OUTPUT_DIR
    / "application_train_clean.parquet"
)

TEST_PARQUET = (
    OUTPUT_DIR
    / "application_test_clean.parquet"
)

TRAIN_CSV = (
    OUTPUT_DIR
    / "application_train_clean.csv"
)

TEST_CSV = (
    OUTPUT_DIR
    / "application_test_clean.csv"
)


# save CSV
train.to_csv(TRAIN_CSV, index=False)
test.to_csv(TEST_CSV, index=False)

print("Saved CSV train:", TRAIN_CSV)
print("Saved CSV test :", TEST_CSV)


# save Parquet
train.to_parquet(TRAIN_PARQUET, index=False)
test.to_parquet(TEST_PARQUET, index=False)

print("Saved Parquet train:", TRAIN_PARQUET)
print("Saved Parquet test :", TEST_PARQUET)

Saved CSV train: ../data/processed/application_train_clean.csv
Saved CSV test : ../data/processed/application_test_clean.csv
Saved Parquet train: ../data/processed/application_train_clean.parquet
Saved Parquet test : ../data/processed/application_test_clean.parquet
